# Forging The Mind Of The Machine: Reading

Through this document, we will take a sample of 70,000 hand drawn numbers, and train a machine learning models to understand it. The goal is to find the best performance through iteration, making changes to find were the sweet spots lay.

The process to teach and train the model will follow these steps:
* Gather baselines
* Data loading, exploring and pre-processing
* Define and experiment with feature selection
* Test and apply defendant ML algorithms, selecting the most efficient(Speed vs Accuracy)
* Model Evaluation for performance
* Tune model parameters for optimal output

Lets begin

## Baseline The Models

Before engineering better ways of doing things, we first must declare a starting point. Using an SGD Classifier model and Design Tree Classifier. We are using classifier models as we have very discrete labels(0-9), and must maintain a clear decision boundary. The model needs to categorize and place new data into the categories, and classifiers are a suitable means to that end, to be more precise multi-class classifiers.

At this point, no hyperparameters or feature engineering will be done, only checking validity in the data and creating a stratified spit. Moving through this document, we will set a baseline for benchmarks; Accuracy, Precision, Recall, Macro F1-score and confusion matrix.

### Visualize and assert the data
Before working with the models, we will pull what we need from the data, and split it and view its shape.

In [1]:
from datetime import time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split


# Download the MNIST dataset
# There is a weird bug in the other parser so we will use the old one: 12/11/2026
mnist = fetch_openml('mnist_784', parser='liac-arff', as_frame=True)

# Check for any missing values within data
print(f"Count of Null rows in Data: {sum(mnist["data"].isnull().sum())}")
# Check and remove duplicates
diff = sum(mnist.data.count())
mnist.data = mnist['data'].drop_duplicates()
diff = diff - sum(mnist.data.count())
print(f"Duplicates removed: {diff}")

# View data
print(mnist.keys())

# Set X as data and y as label
X, y = mnist["data"], mnist["target"]

# Stratify split at 80:20
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y)

print(f"Training set shape: {X_train.shape} {y_train.shape}")
print(f"Testing set shape: {X_test.shape} {y_test.shape}")


Count of Null rows in Data: 0
Duplicates removed: 0
dict_keys(['data', 'target', 'frame', 'categories', 'feature_names', 'target_names', 'DESCR', 'details', 'url'])
Training set shape: (56000, 784) (56000,)
Testing set shape: (14000, 784) (14000,)


With the data ready, we will now push through to the classifier models.

### SGD Classifier
Stochastic Gradient Descent Classifier(SGD), is a linear machine learning model, trained using the Stochastic Gradient Descent algorithm. SGD updates model parameters incrementally using one random training example at a time, creating both fast and memory-efficient training on large datasets, yet still feasible for smaller datasets. On initialization, the model sets random weights assigned to each input feature(SKlearn defaults coefficients to 0), and update in the negative gradient direction to minimize the loss function. This model requires multiple epochs to converge towards optimized weight sets.

The major benefit to using the SGD classifier for the MNIST dataset is the memory and scaling efficiency, as loading 70,000 images in memory would brick up alot of computers, SGD classifier allows us to take on small batches at a time.

As the data has 10 possible outputs, using a multiclass approach is appropriate.


In [6]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import time

# Initialize and train the model
sgd_clf = SGDClassifier(random_state = 42)
print("Training SGD classifier...")
t_start = time.perf_counter()
sgd_clf.fit(X_train, y_train)
t_end = time.perf_counter()

# Training Information
print("Train stats: \n"
      f"\nIterations to converge: {sgd_clf.n_iter_}"
      f"\nTraining time: {(t_end - t_start):.2f} seconds")

# Testing
# First generate predictions based of none labeled test data, model will attempt to class under a existing label from training data
y_pred = sgd_clf.predict(X_test)

# Print individual metrics
print(f"Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"Precision: {precision_score(y_test, y_pred, average="macro")*100:.2f}%")
print(f"Recall: {recall_score(y_test, y_pred, average="macro")*100:.2f}%")
print(f"F1 Score: {f1_score(y_test, y_pred, average="macro")*100:.2f}%")

# Print confusion matrix plus full summary
print(f"Confusion Matrix: \n{confusion_matrix(y_test, y_pred)}")
print(f"Classification Report:\n{classification_report(y_test,y_pred)}")




Training SGD classifier...
Train stats: 

Iterations to converge: 375
Training time: 93.05 seconds
Accuracy: 87.44%
Precision: 88.94%
Recall: 87.00%
F1 Score: 87.02%
Confusion Matrix: 
[[1354    0    1    2    1    0    2    0   17    4]
 [   0 1528   12    2    0    6    3    5   18    1]
 [  18   13 1239   22    6    1   21   13   56    9]
 [  11    8   43 1248    3    7    4    9   80   15]
 [   9   11    6   10 1166    1    8    4   60   90]
 [  33   12   11  101    9  663   22    4  372   36]
 [  17    4    4    1    7    7 1283    0   52    0]
 [   8   12   18    7   21    1    1 1304   25   62]
 [  17   36    8   27    3    4    6    1 1252   11]
 [   8   18    0   16   25    5    0   40   75 1204]]
Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.98      0.95      1381
           1       0.93      0.97      0.95      1575
           2       0.92      0.89      0.90      1398
           3       0.87      0.87      0.87 

### Interpretation

Before diving into the results, an extensions of what each metric is does is to be done:

#### Confusion Matrix
#### Accuracy
#### Precision
#### Recall
#### F1 Score
#### Weighting Score


## Feature Engineering

The data that we will be using the MNIST 784 dataset from SciKits-learns OpenML API. This data set includes 70k gray scale images of hand drawn numbers from 0-9. As we are unsure of the current state of the data we need to asure that we meet the following:

* Cleaning:
    * Handling missing values
    * removing duplicates
    * detecting and handling anomalous data
* Transformation
    * Normalization, scale numerical features to prevent model bias
    * Dimensionality reduction
* Balance data
    * Assure enough data is filled in each feature, making sure not to oversample or undersample maintaining model fairness and accuracy
* Encoding
    * one hot encoding
    * label encoding
    * Creating numerical data out of non-numerical features

It's important to visualize and modify feature, as bugs in the pip-line and unfit data can cause errors or skewed output in the models.

### Loading The Data

In [3]:
from itertools import groupby

from matplotlib.pyplot import axis
from sklearn.datasets import fetch_openml
import matplotlib.pyplot as plt
import pandas as pd

# Download the MNIST dataset
# There is a weird bug in the other parser so we will use the old one: 12/11/2026
mnist = fetch_openml('mnist_784', parser='liac-arff', as_frame=True)

# Check for any missing values within data
print(f"Count of Null rows in Data: {sum(mnist["data"].isnull().sum())}")
# Check and remove duplicates
diff = sum(mnist.data.count())
mnist.data = mnist['data'].drop_duplicates()
diff = diff - sum(mnist.data.count())
print(f"Duplicates removed: {diff}")

# Check and amend Normalization
print(F"Non-Normalized pixel range: {mnist.data.min().min()}-{mnist.data.max().max()}")
mnist.data = (mnist['data'] / mnist.data.max().max())
print(f"Normalized pixel range: {mnist.data.min().min()}-{mnist.data.max().max()}")

# View and assess class balances




Count of Null rows in Data: 0
Duplicates removed: 0
Non-Normalized pixel range: 0.0-255.0
Normalized pixel range: 0.0-1.0
